# 02. Hello World with Vertex AI & Backend Selection

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuichi110/drawlib-examples/blob/main/docs/google-genai/en/02_vertex_ai_quickstart.ipynb)

The core architectural advantage of the unified `google-genai` SDK is that **only the `genai.Client(...)` initialization differs** between Google AI Studio and Google Cloud Vertex AI. Every downstream SDK call—`client.models.generate_content`, streaming, `ThinkingConfig`, Pydantic structured outputs, multi-turn `client.chats`, and tool/function calling—is **100% identical** across both backends.

![Figure 2-1: Unified google-genai SDK Architecture Across AI Studio and Vertex AI](https://examples.drawlib.com/google-genai/images/02_vertex_ai_quickstart/ch02_unified_sdk_arch.png)

*Figure 2-1: Unified google-genai SDK Architecture Across AI Studio and Vertex AI*

---

## 1. Minimal Hello World on Vertex AI (Keyless IAM Auth in Colab)

When targeting Google Cloud Vertex AI, you do not need to generate or distribute static API keys. Inside Google Colab, `auth.authenticate_user()` opens a browser sign-in prompt to establish short-lived Application Default Credentials (ADC) backed by Google Cloud IAM. Ensure the Vertex AI API (`aiplatform.googleapis.com`) is enabled on your project and your Google account has the **Vertex AI User** (`roles/aiplatform.user`) role.

In [ ]:
!pip install -q drawlib google-genai pydantic

In [ ]:
from google.colab import auth
from google import genai

# Authenticate with your Google account in the browser popup
auth.authenticate_user()

client = genai.Client(
    vertexai=True,
    project="your-gcp-project-id",  # Replace with your Google Cloud Project ID
    location="us-central1",         # Or "asia-northeast1", "global", etc.
)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Hello from Vertex AI! List three advantages of keyless IAM authentication in two sentences.",
)
print(response.text)

### Keeping Your Project ID Out of Shared Notebooks

When committing notebooks to Git or sharing them with teammates, avoid hardcoding your Google Cloud Project ID in source cells. Store `GOOGLE_CLOUD_PROJECT` in **Colab Secrets** (the key icon in the left sidebar, with **Notebook access** toggled on) and read it dynamically with `userdata.get("GOOGLE_CLOUD_PROJECT")`:

In [ ]:
from google.colab import auth, userdata
from google import genai

auth.authenticate_user()

# Load GOOGLE_CLOUD_PROJECT securely from Colab Secrets
client = genai.Client(
    vertexai=True,
    project=userdata.get("GOOGLE_CLOUD_PROJECT"),
    location="asia-northeast1",
)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Confirm that Vertex AI initialized cleanly using Colab Secrets.",
)
print(response.text)

> **Regional Endpoints vs. `location="global"`:**
> Use a **regional endpoint** such as `"us-central1"` (Iowa) or `"asia-northeast1"` (Tokyo) when your workload requires strict geographic data residency and predictable regional routing. Use **`location="global"`** when you want dynamic multi-region routing for maximum quota availability—or when testing **bleeding-edge Preview models** on Vertex AI that launch exclusively on the `global` endpoint before rolling out to individual regions.

---

## 2. Choosing Between Google AI Studio and Vertex AI

Because `google-genai` unifies both platforms behind one client interface, you can start prototyping in minutes with Google AI Studio and transition to Vertex AI whenever enterprise governance requires it—without rewriting a single model or chat call.

![Figure 2-2: Decision Matrix — Google AI Studio vs. Google Cloud Vertex AI](https://examples.drawlib.com/google-genai/images/02_vertex_ai_quickstart/ch02_backend_comparison.png)

*Figure 2-2: Decision Matrix — Google AI Studio vs. Google Cloud Vertex AI*

### How Chapters 3 Through 5 Work

- **Concise Default Initialization in Chapters 3–5**: Because Google AI Studio API keys can be linked directly to a Google Cloud Project for billing and higher rate limits (as covered in Chapter 1), **Chapters 3 through 5 standardize on the concise 3-line AI Studio initialization** (`genai.Client(api_key=userdata.get("GEMINI_API_KEY"))`) so every notebook stays minimal and focused on core SDK capabilities.
- **Drop-In Compatibility for Vertex AI Users**: Every single code cell in Chapters 3–5—covering Streaming, `ThinkingConfig`, Multimodal Inputs, Pydantic Structured Outputs, Multi-turn Chat, and Function Calling—**runs unchanged on Vertex AI** simply by replacing that 3-line `client` initialization with this chapter's `vertexai=True` setup.